In [1]:
# Imports

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score , accuracy_score

# sys.path.append(os.path.join(".."))
# sys.path.append(os.path.join("../src"))

# from preprocessing import get_X_y , get_X

In [2]:
# Data Load

df_train = pd.read_csv("../dataset/train.csv")
df_test = pd.read_csv("../dataset/test.csv")

df_train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [6]:
# Null Values

print(df_train.isnull().sum())

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [7]:
def get_X_y(df) :

    df[["Group" , "pass_no"]] = df["PassengerId"].str.split("_" , expand = True)

    df["Expense"] = df["RoomService"] + df["FoodCourt"] + df["ShoppingMall"] + df["Spa"] + df["VRDeck"]

    expense_column = ["RoomService"	,
                    "FoodCourt" ,
                    "ShoppingMall" ,
                    "Spa" ,
                    "VRDeck"
    ]

    for i in expense_column :
        df.loc[(df["CryoSleep"] == True) & (df[i].isnull()) , i] = 0.0
    
    df.loc[(((df["HomePlanet"] == "Europa") | (df["HomePlanet"] == "Mars")) & (df["VIP"].isnull())) , "VIP"] = True
    df.loc[((df["HomePlanet"] == "Earth") & (df["VIP"].isnull())) , "VIP"] = False

    df.loc[((df["VIP"] == False) & (df["HomePlanet"].isnull())) , "HomePlanet"] = "Earth"

    df.loc[((df["Expense"] == 0.0) & (df["CryoSleep"].isnull())) , "CryoSleep"] = True
    df.loc[((df["Expense"] != 0.0) & (df["CryoSleep"].isnull())) , "CryoSleep"] = False

    for i in expense_column :
        df.loc[((df["VIP"] == True) & (df[i].isnull())) , i] = float((df[df["VIP"] == True])[i].median())

    for i in expense_column :
        df.loc[((df["VIP"] == False) & (df[i].isnull())) , i] = float((df[df["VIP"] == False])[i].median())

    unique_group = df["Group"].unique().tolist()

    for i in unique_group:
        cabin = df.loc[
            (df["Group"] == i) & (df["Cabin"].notnull()),
            "Cabin"
        ]

        if not cabin.empty:
            df.loc[
                (df["Group"] == i) & (df["Cabin"].isnull()),
                "Cabin"
            ] = cabin.iloc[0]

    df = df.dropna(subset = ["Cabin"])

    # df["Age"] = df["Age"].fillna(df["Age"].median())

    # df["Destination"] = df["Destination"].fillna(df["Destination"].mode()[0])

    df[["Deck", "CabinNum", "Side"]] = df["Cabin"].str.split("/", expand=True)

    df = df.drop(["Name"  , "Cabin"  , "Expense"] , axis = 1)

    df["CabinNum"] = df["CabinNum"].values.astype(int)
    df["Group"] = df["Group"].values.astype(int)

    X = df.drop(["Transported"] , axis = 1)
    y = df["Transported"]

    y = y.astype(int)

    null_planet_id = X["PassengerId"][X["HomePlanet"].isnull()].values
    
    for i in null_planet_id :
    
        if ((((X["VIP"][X["PassengerId"] == i]).values[0]) == True) | (((X["VIP"][X["PassengerId"] == i]).values[0]) == False)) :
            X.loc[(X["PassengerId"] == i) , "HomePlanet"] = X.loc[
                                                            (X["Deck"] == ((X["Deck"][X["PassengerId"] == i]).values[0])) & 
                                                            (X["Side"] == ((X["Side"][X["PassengerId"] == i]).values[0])) &
                                                            (X["pass_no"] == ((X["pass_no"][X["PassengerId"] == i]).values[0])) &
                                                            (X["CryoSleep"] == ((X["CryoSleep"][X["PassengerId"] == i]).values[0])) &
                                                            (X["VIP"] == ((X["VIP"][X["PassengerId"] == i]).values[0]))
                                                            , "HomePlanet"].mode().values[0]
        else :
            
            X.loc[(X["PassengerId"] == i) , "HomePlanet"] = X.loc[
                                                            (X["Deck"] == ((X["Deck"][X["PassengerId"] == i]).values[0])) & 
                                                            (X["Side"] == ((X["Side"][X["PassengerId"] == i]).values[0])) &
                                                            (X["pass_no"] == ((X["pass_no"][X["PassengerId"] == i]).values[0])) &
                                                            (X["CryoSleep"] == ((X["CryoSleep"][X["PassengerId"] == i]).values[0]))
                                                            , "HomePlanet"].mode().values[0]

    X.loc[(((X["HomePlanet"] == "Europa") | (X["HomePlanet"] == "Mars")) & (X["VIP"].isnull())) , "VIP"] = True
    X.loc[((X["HomePlanet"] == "Earth") & (X["VIP"].isnull())) , "VIP"] = False

    X = X.drop(["PassengerId"] , axis = 1)

    X["pass_no"] = X["pass_no"].values.astype(int)
    
    return X , y

In [8]:
X , y = get_X_y(df_train)

In [363]:
X

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,pass_no,Deck,CabinNum,Side
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,1,1,B,0,P
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,2,1,F,0,S
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,3,1,A,0,S
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,3,2,A,0,S
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,4,1,F,1,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,9276,1,A,98,P
8689,Earth,True,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,9278,1,G,1499,S
8690,Earth,False,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,9279,1,G,1500,S
8691,Europa,False,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,9280,1,E,608,S


In [365]:
print(X.isnull().sum())

HomePlanet        0
CryoSleep         0
Destination     180
Age             175
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Group             0
pass_no           0
Deck              0
CabinNum          0
Side              0
dtype: int64


In [19]:
X[X["Destination"].isnull()]

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,pass_no,Deck,CabinNum,Side
47,Mars,True,NaN,19.0,False,0.0,0.0,0.0,0.0,0.0,45,2,F,10,P
128,Earth,False,NaN,34.0,False,0.0,22.0,0.0,564.0,207.0,138,2,E,5,P
139,Earth,False,NaN,41.0,False,0.0,0.0,0.0,0.0,607.0,152,1,F,32,P
347,Earth,False,NaN,23.0,False,348.0,0.0,0.0,4.0,368.0,382,1,G,64,P
430,Earth,True,NaN,50.0,False,0.0,0.0,0.0,0.0,0.0,462,1,G,67,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8372,Earth,True,NaN,20.0,False,0.0,0.0,0.0,0.0,0.0,8956,2,G,1453,P
8551,Mars,True,NaN,41.0,False,0.0,0.0,0.0,0.0,0.0,9130,1,F,1765,S
8616,Mars,True,NaN,33.0,False,0.0,0.0,0.0,0.0,0.0,9195,2,F,1779,S
8621,Europa,False,NaN,41.0,True,0.0,7964.0,0.0,3238.0,5839.0,9197,2,C,308,P


In [37]:
# X[(X["Deck"] == "F") & (X["Side"] == "P") & ].head(50)

X[(X["pass_no"] == 2) & (X["Side"] == "P")].head(50)

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,pass_no,Deck,CabinNum,Side
10,Europa,True,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,8,2,B,1,P
20,Earth,False,55 Cancri e,14.0,False,412.0,0.0,1.0,0.0,679.0,17,2,F,6,P
34,Mars,False,TRAPPIST-1e,2.0,False,0.0,0.0,0.0,0.0,0.0,31,2,F,9,P
44,Earth,True,55 Cancri e,4.0,False,0.0,0.0,0.0,0.0,0.0,44,2,G,3,P
47,Mars,True,NaN,19.0,False,0.0,0.0,0.0,0.0,0.0,45,2,F,10,P
76,Mars,True,TRAPPIST-1e,2.0,False,0.0,0.0,0.0,0.0,0.0,82,2,F,16,P
86,Earth,True,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,92,2,G,9,P
90,Earth,False,TRAPPIST-1e,26.0,False,0.0,2811.0,957.0,0.0,87.0,98,2,G,11,P
92,Earth,True,TRAPPIST-1e,2.0,False,0.0,0.0,0.0,0.0,0.0,99,2,G,12,P
104,Europa,False,TRAPPIST-1e,40.0,False,0.0,331.0,0.0,0.0,1687.0,110,2,B,5,P
